# Australia Petroleum Statistics — Phase 1 (v0 scraper)

**Goal**: download the latest monthly *Australian Petroleum Statistics* Excel file from the energy.gov.au publications page so we have something to parse and dashboard in later phases.

**Source page**: https://www.energy.gov.au/publications/australian-petroleum-statistics-2026

**Why a notebook for this**: it lets us pause between steps and poke at intermediate values (`resp`, `soup`, `xlsx_links`, the raw bytes) — exactly what's useful when you're learning what each library actually does. The companion script `scripts/scrape_australia_v0.py` has the same logic, condensed to run from the command line.

### Strategy in one paragraph

1. `GET` the publications page HTML.
2. Parse it with BeautifulSoup, find the **first** `<a href="…xlsx">`. The page lists the latest monthly extract at the top under "Attachments"; older yearly publications appear lower on the page but link to *other* publication pages (not directly to xlsx files), so a "first .xlsx link" rule is enough.
3. Download the file and **verify it's really an xlsx** before writing — check both the `Content-Type` header (informational) and the file's magic bytes (authoritative).
4. Save under the original filename in `data/raw/australia/`, overwriting any previous copy.

> **Note on `curl_cffi`**: our first attempt with plain `requests` got silently stalled by energy.gov.au's bot protection (a WAF doing TLS fingerprinting — it recognised Python's TLS handshake as non-browser and refused to respond). We diagnosed this with `scripts/_diagnose_network.py` (only energy.gov.au failed; all other gov + commercial sites worked), then confirmed `curl_cffi` defeats it with `scripts/_try_curl_cffi.py`. So this notebook uses `curl_cffi`, a drop-in `requests` replacement that performs the TLS handshake byte-for-byte like Chrome. The WAF can no longer tell us apart from a real browser.

In Phase 2 we'll wrap this in a class that inherits from `scrapers.base.BaseScraper`, just like `scrapers/india_ppac.py`.

## 1. Imports

- `curl_cffi.requests` — a `requests`-compatible client that performs the TLS handshake using Chrome's exact algorithm ordering. We use it here (rather than plain `requests`) because energy.gov.au's WAF blocks Python's default TLS fingerprint. The rest of the codebase still uses plain `requests`; this notebook is the exception.
- `BeautifulSoup` from `bs4` — HTML parser (same as `scrapers/india_ppac.py`).
- `pathlib.Path` — cross-platform path arithmetic. We use it instead of raw strings + `os.path.join` because it's much harder to typo a separator with `/` as an operator.

In [1]:
from datetime import datetime
from pathlib import Path

from curl_cffi import requests
from bs4 import BeautifulSoup

## 2. Constants — and the impersonation profile

For most government / CDN-fronted sites a custom **User-Agent header** is enough to bypass simple bot filters. Energy.gov.au is stricter — it inspects the TLS handshake itself, so a fake User-Agent on top of Python's real TLS fingerprint isn't enough. We need `curl_cffi`'s impersonation. Picking a profile is just naming which browser+version we want to pretend to be:

- `IMPERSONATE_PROFILE = "chrome131"` — `curl_cffi` performs the TLS handshake byte-for-byte like Chrome 131, and automatically sets a matching set of HTTP headers (User-Agent, Accept, Accept-Language, Sec-Fetch-*, etc.). We don't have to hand-roll any headers ourselves — passing `headers=...` on top of impersonation would actually *override* the realistic ones curl_cffi sets, so we just leave it off.

The other constants are standard:
- `PAGE_URL_TEMPLATE` — a format string with `{year}` in it. The publications page URL contains the calendar year (e.g. `...australian-petroleum-statistics-2026`), so we build the actual URL each run rather than hardcoding a year that would silently go stale on January 1st.
- `BASE_URL` — used to turn relative hrefs (`/sites/default/files/...`) into absolute URLs.
- `XLSX_MAGIC` — the first 4 bytes of every xlsx file. An xlsx is actually a zip archive of XML files, so its magic bytes are the ZIP signature `b"PK\x03\x04"`. We'll use this as our authoritative check that the downloaded bytes are really an xlsx.
- `RAW_DIR` — where the downloaded file ends up on disk.

### And a `find_publications_url()` helper

To survive the year transition, we don't just plug `datetime.now().year` into the template — that would silently 404 in (e.g.) January 2027 if DCCEEW hasn't yet launched the `-2027` page. Instead, the helper:

1. Tries this calendar year first.
2. Falls back to last year if the current-year page returns a non-200 status.
3. Raises a clear `RuntimeError` if both fail.

It returns **both** the URL and the response object so cell 6 doesn't have to fetch the same page twice. Taking `today` as a parameter (default `datetime.now()`) makes the function easy to test: `find_publications_url(today=datetime(2027, 1, 15))` simulates next January's edge case without waiting.

In [2]:
PAGE_URL_TEMPLATE = "https://www.energy.gov.au/publications/australian-petroleum-statistics-{year}"
BASE_URL = "https://www.energy.gov.au"

# Which Chrome version's TLS handshake curl_cffi should impersonate.
# We confirmed "chrome131" defeats energy.gov.au's WAF via
# scripts/_try_curl_cffi.py.
IMPERSONATE_PROFILE = "chrome131"

XLSX_MAGIC = b"PK\x03\x04"


def find_publications_url(today: datetime | None = None) -> tuple[str, "requests.Response"]:
    """
    Find the working publications-page URL by trying this calendar year first
    and falling back to the previous year.

    The publication is monthly and the page URL contains the calendar year, so
    we try `{current_year}` first then `{current_year - 1}` to bridge the gap
    where DCCEEW hasn't yet launched the new-year page (typically January each
    year). Two candidates are enough: by February the current year is always
    live. Returning the Response saves a second GET in cell 6.

    Raises RuntimeError listing what was tried if neither candidate returns 200.
    """
    today = today or datetime.now()
    candidates = [today.year, today.year - 1]
    last_error = None
    for year in candidates:
        url = PAGE_URL_TEMPLATE.format(year=year)
        print(f"  Trying year={year}: {url}")
        try:
            resp = requests.get(url, timeout=30, impersonate=IMPERSONATE_PROFILE)
            if resp.ok:
                print(f"  → OK ({resp.status_code})")
                return url, resp
            last_error = f"HTTP {resp.status_code}"
            print(f"  → {last_error}, trying next…")
        except Exception as e:
            last_error = f"{type(e).__name__}: {e}"
            print(f"  → FAILED ({last_error}), trying next…")
    raise RuntimeError(
        f"No working publications URL found.\n"
        f"  Tried years: {candidates}\n"
        f"  Last error : {last_error}"
    )

# Resolve the project root by walking up from this notebook's directory.
# Same pattern used in notebooks/05_jodi_dashboard.ipynb so the path works
# whether Jupyter was started from country_oil_scraper/ or one level up.
def _resolve_project_root() -> Path:
    here = Path.cwd()
    for candidate in [here, *here.parents]:
        if (candidate / "scrapers" / "base.py").exists():
            return candidate
        if (candidate / "country_oil_scraper" / "scrapers" / "base.py").exists():
            return candidate / "country_oil_scraper"
    raise RuntimeError(f"Could not locate project root from cwd: {here}")

PROJECT_ROOT = _resolve_project_root()
RAW_DIR = PROJECT_ROOT / "data" / "raw" / "australia"
print(f"Project root: {PROJECT_ROOT}")
print(f"Raw dir     : {RAW_DIR}")

Project root: c:\Users\luiscarlos.gaitan\OneDrive - Jain Global\Coding\country_oil_scraper
Raw dir     : c:\Users\luiscarlos.gaitan\OneDrive - Jain Global\Coding\country_oil_scraper\data\raw\australia


## 3. Fetch the publications page

We call `find_publications_url()` which does the discovery (current year, falling back to previous year) and returns both the working URL and the response object. That way we don't pay for two GETs to the same page.

Two general patterns to remember about `requests.get`:

- `timeout=30` means "fail loudly if no headers arrive within 30s". Without a timeout, a flaky network can hang the script forever.
- `resp.raise_for_status()` raises `HTTPError` on 4xx/5xx. Always call it after a `GET` — otherwise a 403 just gives you a tiny error-page HTML body and your parsing later fails with a cryptic IndexError. (We call it here belt-and-braces, even though `find_publications_url` already filters to `.ok` responses.)

**Try this**: simulate next January's edge case with `find_publications_url(today=datetime(2027, 1, 15))`. You'll see the function try `-2027` first, fall back to `-2026`, and succeed — without changing your system clock.

In [3]:
PAGE_URL, resp = find_publications_url()
resp.raise_for_status()

print()
print(f"Page URL    : {PAGE_URL}")
print(f"Status code : {resp.status_code}")
print(f"Content-Type: {resp.headers.get('Content-Type')}")
print(f"HTML length : {len(resp.text):,} chars")
print(f"First 200   : {resp.text[:200]!r}")

  Trying year=2026: https://www.energy.gov.au/publications/australian-petroleum-statistics-2026
  → OK (200)

Page URL    : https://www.energy.gov.au/publications/australian-petroleum-statistics-2026
Status code : 200
Content-Type: text/html; charset=UTF-8
HTML length : 50,671 chars
First 200   : '<!DOCTYPE html>\n<html lang="en" dir="ltr">\n  <head>\n    <meta charset="utf-8" />\n<script async src="https://www.googletagmanager.com/gtag/js?id=G-6QBCSR4HZN"></script>\n<script>window.dataLayer = windo'


## 4. Parse the HTML and list every xlsx link

`BeautifulSoup` builds a tree from the HTML so we can query it like a DOM. We use the `lxml` parser (already in `requirements.txt`) because it's the fastest of the BS4 backends.

`soup.find_all("a", href=True)` walks the whole tree and yields every `<a>` tag with a non-empty `href`. The list comprehension then keeps only those whose href ends in `.xlsx` (case-insensitively — government CMSes are sometimes inconsistent).

**Try this**: print `soup.prettify()[:2000]` in a scratch cell to see what the HTML actually looks like. Or try `[a.get_text(strip=True) for a in soup.find_all('a', href=True) if '.xlsx' in a['href'].lower()]` to see the human-readable link text next to each xlsx URL.

In [10]:
import lxml

soup = BeautifulSoup(resp.text, "lxml")

xlsx_links = [
    a["href"]
    for a in soup.find_all("a", href=True)
    if a["href"].lower().endswith(".xlsx")
]

print(f"Found {len(xlsx_links)} xlsx link(s):")
for i, h in enumerate(xlsx_links):
    print(f"  [{i}] {h}")

Found 2 xlsx link(s):
  [0] /sites/default/files/2026-04/australian_petroleum_statistics_-_data_extract_february_2026.xlsx
  [1] /sites/default/files/2026-03/Australian%20Petroleum%20Statistics%20-%20Data%20Extract%20January%202026.xlsx


## 5. Pick the first link and build an absolute URL

On this page the **first** xlsx link is the latest monthly extract — the page lists it at the top under "Attachments", and the other yearly publications on the page link to other pages (not xlsx files), so they never appear in our filter.

The href from the HTML is relative (`/sites/default/files/...`), so we have to prepend `BASE_URL` before passing it to `requests.get` — relative URLs only mean something to a browser that already knows the current page's host.

In [22]:
if not xlsx_links:
    raise RuntimeError(f"No .xlsx links on {PAGE_URL} — page layout may have changed.")

href = xlsx_links[0]
url = href if href.startswith("http") else BASE_URL + href

print(f"href : {href}")
print(f"url  : {url}")

href : /sites/default/files/2026-04/australian_petroleum_statistics_-_data_extract_february_2026.xlsx
url  : https://www.energy.gov.au/sites/default/files/2026-04/australian_petroleum_statistics_-_data_extract_february_2026.xlsx


## 6. Download and verify it's really an xlsx

Two checks before we trust the bytes:

1. **Content-Type header** — informational only. We log it and warn if it doesn't look spreadsheet-y, but we don't abort on it. Some servers serve xlsx as `application/octet-stream` or even `text/html`; the header is a sanity hint, not a guarantee.
2. **Magic bytes** — authoritative. Every xlsx file starts with the ZIP local-file header signature `b"PK\x03\x04"` (because an xlsx is literally a zip archive). If those four bytes aren't there, we definitely didn't get an xlsx — most likely we got an HTML error page that the server mislabelled.

A small `curl_cffi` quirk vs. plain `requests`: the `Response` object **is not** a context manager. So you can't write `with requests.get(url) as r:` like you can with the regular `requests` library — you'd get `TypeError: 'Response' object does not support the context manager protocol`. For our 2-3 MB file that's fine; we just call `requests.get(...)` and read `r.content` directly. (In `requests`, `stream=True` plus a `with` block lets you read response headers before deciding whether to pull the body — a memory win for very large files. `curl_cffi` has its own `stream=True` mechanic but it's API-different; we don't need it here.)

**Try this**: change the URL to something bogus (e.g. add `_typo` before `.xlsx`) and re-run — you should see the magic-bytes check fire.

In [23]:
r = requests.get(url, timeout=120, impersonate=IMPERSONATE_PROFILE)
r.raise_for_status()
content_type = r.headers.get("Content-Type", "<unset>").lower()
body = r.content

# Soft check — warn but don't fail
spreadsheet_hints = ("spreadsheet", "excel", "openxml", "octet-stream")
if not any(h in content_type for h in spreadsheet_hints):
    print(f"[!] Content-Type {content_type!r} doesn't look spreadsheet-y — continuing.")

# Hard check — must be a zip archive
if not body.startswith(XLSX_MAGIC):
    raise RuntimeError(
        f"Downloaded {len(body)} bytes but they aren't an xlsx.\n"
        f"  Content-Type : {content_type}\n"
        f"  First 16 bytes: {body[:16]!r}"
    )

print(f"Size         : {len(body) / 1024:.0f} KB")
print(f"Content-Type : {content_type}")
print(f"Magic bytes  : {body[:4]!r}  (OK)")

Size         : 2656 KB
Content-Type : application/vnd.openxmlformats-officedocument.spreadsheetml.sheet
Magic bytes  : b'PK\x03\x04'  (OK)


## 7. Save under the original filename

`Path(href).name` strips the directory portion of the href and gives us just the filename, e.g. `australian_petroleum_statistics_-_data_extract_february_2026.xlsx`. We keep the original name so it's obvious at a glance which month the file refers to.

`mkdir(parents=True, exist_ok=True)` creates the entire directory chain if it's missing and is a no-op if it already exists — much cleaner than a `try/except FileExistsError`.

`write_bytes` overwrites any previous copy at that path. For this learning phase that's deliberate — we always pull the latest, and don't keep snapshots. Phase 2's class version will switch to timestamped filenames (`petroleum_statistics_<YYYYMMDD>.xlsx`) so we can keep a history.

In [24]:
filename = Path(href).name
RAW_DIR.mkdir(parents=True, exist_ok=True)

out_path = RAW_DIR / filename
out_path.write_bytes(body)

print(f"Saved → {out_path}")
print(f"        {out_path.stat().st_size / 1024:.0f} KB on disk")

Saved → c:\Users\luiscarlos.gaitan\OneDrive - Jain Global\Coding\country_oil_scraper\data\raw\australia\australian_petroleum_statistics_-_data_extract_february_2026.xlsx
        2656 KB on disk


## Done.

You now have the latest `australian_petroleum_statistics_-_data_extract_*.xlsx` saved locally and can open it in Excel or `pandas.read_excel(out_path)` to see what's inside.

**What's worth playing with before we move on:**

1. Try printing `soup.prettify()[:2000]` to see what HTML the page actually returns.
2. Inspect `xlsx_links` more carefully — what other URLs would `.find_all("a", href=True)` return without the `.xlsx` filter?
3. Open the downloaded file: `import pandas as pd; xls = pd.ExcelFile(out_path); print(xls.sheet_names)` — 26 sheets covering production, refining, sales, imports, exports.
4. Run the matching script: `python scripts/scrape_australia_v0.py` from the project root. Same logic, different shape.

When you're ready, say **"ready for phase 2"** and we'll refactor this into a `BaseScraper` subclass and wire it into `config/sources.yaml`.